# Day 7 — Embeddings + Semantic Search

In [1]:
!pip install chromadb sentence-transformers -q
print("Installation complete")

Installation complete


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
print('All libraries imported successfully')
print(f'ChromaDB version : {chromadb.__version__}')

All libraries imported successfully
ChromaDB version : 1.5.9


In [3]:
documents =[
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automobiles",
    "SQL is used to query databases",
    "Machine learning trains models on data",
]

query_keyword = "vehicle"
print("="*60)
print(f"KEYWORS SEARCH for:'{query_keyword}'")
print("="*60)
for i, doc in enumerate(documents):
  if query_keyword.lower() in doc.lower():
    print(f'FOUND [doc_{i}]:{doc}')
  else:
    print(f'MISSED[doc_{i}]:{doc}')


print()
print("PEOPLE: doc_2 talks about 'Cars and tracks' - which ARE vehicles")
print("But keyword search missed it because it searched for the exact word 'vehicle'")

KEYWORS SEARCH for:'vehicle'
MISSED[doc_0]:ETL is used to clean and transform data
FOUND [doc_1]:A vehicle is a mode of transportation
MISSED[doc_2]:Cars and trucks are popular automobiles
MISSED[doc_3]:SQL is used to query databases
MISSED[doc_4]:Machine learning trains models on data

PEOPLE: doc_2 talks about 'Cars and tracks' - which ARE vehicles
But keyword search missed it because it searched for the exact word 'vehicle'


In [4]:
failure_examples=[
     {"query": "I feel sick",         "misses": "I am unwell, patient has fever"},
    {"query": "How to cook rice",    "misses": "Steps to prepare rice"},
    {"query": "vehicle speed",       "misses": "car acceleration, automobile velocity"},
    {"query": "ML model accuracy",   "misses": "classification performance, prediction quality"},
]

print("KEYWORD SEARCH FAILURE CASES")
print("="*60)
for ex in failure_examples:
  print(f"Query:'{ex['query']}'")
  print(f"Misses:{ex['misses']}")
  print("-"*40)
print()
print("SOLUTION: We need search that understands MEANING,not just characters")
print("That is what EMBEDDINGS do")

KEYWORD SEARCH FAILURE CASES
Query:'I feel sick'
Misses:I am unwell, patient has fever
----------------------------------------
Query:'How to cook rice'
Misses:Steps to prepare rice
----------------------------------------
Query:'vehicle speed'
Misses:car acceleration, automobile velocity
----------------------------------------
Query:'ML model accuracy'
Misses:classification performance, prediction quality
----------------------------------------

SOLUTION: We need search that understands MEANING,not just characters
That is what EMBEDDINGS do


Vector - A list of numbers for embeddings ar list of 384(numbers)

Dimension - One number in the list (384 dimensions - 384 numbers)

Cosine Similarity - A score from 0.0 to 1.0 - how similar two vectors are

Embeddings Model - The neural network that converts text to vectors

In [ ]:
print("Loading embedding model...(may take 1-2 minutes on first run)")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully!")
print(f'Model produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions')

In [6]:
sentence = "ETL is used to clean and transform data"
embedding = model.encode(sentence)
print(f"Input sentence :{sentence}")
print()
print(f"Embedding type: {type(embedding)}")
print(f"Embedding shape:{embedding.shape}")
print(f"First 10 numbers: {embedding[:10].round(4)}")
print(f"Min Value: {embedding.min():.4f}")
print(f"Max Value: {embedding.max():.4f}")



Input sentence :ETL is used to clean and transform data

Embedding type: <class 'numpy.ndarray'>
Embedding shape:(384,)
First 10 numbers: [-0.0784  0.0541  0.0224 -0.0389  0.0221 -0.0904  0.0007 -0.0152  0.0733
  0.0362]
Min Value: -0.1381
Max Value: 0.1815


In [7]:
sentences = [
    "ETL is used to clean and transform data",
  "Data Transformation is a key pipeline step",
  "The sky is blue and clouds are white"
]

embeddings = model.encode(sentences)


In [8]:
print(f"Number of sentences: {len(sentences)}")
print(f"Shape of embeddings array: {embeddings.shape}")

print()
print("Each row is one sentence's embedding:")
for i, sent in enumerate(sentences):
  print(f"Sentence {i}:shape={embeddings[i].shape},First 5 values:{embeddings[i][:5].round(3)}")


Number of sentences: 3
Shape of embeddings array: (3, 384)

Each row is one sentence's embedding:
Sentence 0:shape=(384,),First 5 values:[-0.078  0.054  0.022 -0.039  0.022]
Sentence 1:shape=(384,),First 5 values:[-0.047  0.059 -0.001 -0.033 -0.046]
Sentence 2:shape=(384,),First 5 values:[0.054 0.06  0.075 0.049 0.05 ]


In [9]:
def cosine_similarity(vec_a,vec_b):
  dot_product = np.dot(vec_a,vec_b)
  norm_a = np.linalg.norm(vec_a)
  norm_b = np.linalg.norm(vec_b)
  return dot_product/(norm_a*norm_b)
sim_01 = cosine_similarity(embeddings[0],embeddings[1])
sim_02 = cosine_similarity(embeddings[0],embeddings[2])
sim_03 = cosine_similarity(embeddings[1],embeddings[2])

print("COSINE SIMILARITY SCORES")
print("="*60)
print(f"Sentence 0:'{sentence[0]}'")
print(f"Sentence 1:'{sentence[1]}'")
print(f"Sentence 2:'{sentence[2]}'")
print()
print(f"Similarity (0 vs 1):{sim_01:.4f} <- Expected: HIGH (same topic)")
print(f"Similarity (0 vs 2):{sim_01:.4f} <- Expected: LOW (different topic)")
print(f"Similarity (1 vs 2):{sim_01:.4f} <- Expected: LOW (different topic)")
print()
print("INSIGHT: Sentences 0 and 1 have different words but similar meaning.")
print("Their cosne similarity score is high - the embedding captured the meaning")

COSINE SIMILARITY SCORES
Sentence 0:'E'
Sentence 1:'T'
Sentence 2:'L'

Similarity (0 vs 1):0.4357 <- Expected: HIGH (same topic)
Similarity (0 vs 2):0.4357 <- Expected: LOW (different topic)
Similarity (1 vs 2):0.4357 <- Expected: LOW (different topic)

INSIGHT: Sentences 0 and 1 have different words but similar meaning.
Their cosne similarity score is high - the embedding captured the meaning


In [10]:
your_sentences =[
    "Machine learning trains models on labeled data",
    "AI algorithm learn patterns from example",
    "I enjoy eating pizza for lunch"
]
your_embeddings = model.encode(your_sentences)
sim_your_01 = cosine_similarity(your_embeddings[0],your_embeddings[1])
sim_your_02 = cosine_similarity(your_embeddings[0],your_embeddings[2])

print("YOUR EXPERIMENT RESULTS")
print("="*60)
print(f"Sentence A:'{your_sentences[0]}")
print(f"Sentence B:'{your_sentences[1]}")
print(f"Sentence C:'{your_sentences[2]}")
print()
print(f"Similarity (A vs B): {sim_your_01:.4f}")
print(f"Similarity (A vs C): {sim_your_02:.4f}")
print()

if sim_your_01 > sim_your_02:
  print("Correct: A and B are more similar to each other that A and C")
else:
  print("Interesting! A and C ended up more similar. Try adjusting your sentences.")

YOUR EXPERIMENT RESULTS
Sentence A:'Machine learning trains models on labeled data
Sentence B:'AI algorithm learn patterns from example
Sentence C:'I enjoy eating pizza for lunch

Similarity (A vs B): 0.3085
Similarity (A vs C): -0.0452

Correct: A and B are more similar to each other that A and C


In [11]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("demo_notes")
print("ChromaDB client created (in-memory mode)")
print(f"Collection name: demo_notes")
print(f"Documents in collection: {collection.count()}")


ChromaDB client created (in-memory mode)
Collection name: demo_notes
Documents in collection: 0


In [12]:
sample_docs=[
    "ETL stands for Extract Transform Load — the core data engineering process",
    "SQL SELECT statements retrieve data from database tables",
    "Machine learning models learn patterns from training data",
    "Python Pandas library is used for data manipulation and cleaning",
    "Neural networks are inspired by how the human brain works",
]
sample_ids=["doc001","doc002","doc003","doc004","doc005"]
sample_metadata=[
   {"subject": "Data Engineering", "topic": "ETL"},
    {"subject": "Data Engineering", "topic": "SQL"},
    {"subject": "Machine Learning", "topic": "ML Basics"},
    {"subject": "Python",           "topic": "Pandas"},
    {"subject": "Machine Learning", "topic": "Neural Networks"},
]
collection.add(
     documents=sample_docs,
     metadatas=sample_metadata,
     ids=sample_ids
 )


In [13]:
print(f"Documents added to collection:")
print(f"Total documents now in collection: {collection.count()}")

Documents added to collection:
Total documents now in collection: 5


In [14]:
query ="How do I clean and prepare data?"
results = collection.query(
    query_texts=[query], n_results = 3
)
print("RESULT KEY AVAILABLE:")
print(list(results.keys()))

RESULT KEY AVAILABLE:
['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']


In [15]:
print(f"Query:'{query}")
print("="*50)
print()
matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_distances = results['distances'][0]
matched_metadata = results['metadatas'][0]

for rank,(doc,doc_id, dist, meta) in enumerate(zip(matched_docs, matched_ids, matched_distances, matched_metadata),start=1):
  print(f"Rank {rank} | ID: {doc_id} | Distance : {dist:.4f}")
  print(f"Subject: {meta["subject"]}| Topic : {meta['topic']}")
  print(f"Document: {doc}")
  print()
print("NOTICE: The results are about ETL and Pandas — exactly what 'clean and prepare data' means!")
print("Semantic search found them even though the words are different.")

Query:'How do I clean and prepare data?

Rank 1 | ID: doc004 | Distance : 1.1045
Subject: Python| Topic : Pandas
Document: Python Pandas library is used for data manipulation and cleaning

Rank 2 | ID: doc002 | Distance : 1.6139
Subject: Data Engineering| Topic : SQL
Document: SQL SELECT statements retrieve data from database tables

Rank 3 | ID: doc003 | Distance : 1.6540
Subject: Machine Learning| Topic : ML Basics
Document: Machine learning models learn patterns from training data

NOTICE: The results are about ETL and Pandas — exactly what 'clean and prepare data' means!
Semantic search found them even though the words are different.


In [16]:
filtered_query = "how do computers learn from examples"

filtered_results = collection.query(
    query_texts=[filtered_query],
    n_results=3,
    where={"subject": "Machine Learning"}
)

print(f"Filtered Query: '{filtered_query}'")
print("Filter: Only Machine Learning documents")
print("=" * 60)

for rank, (doc, dist, meta) in enumerate(
    zip(
        filtered_results["documents"][0],
        filtered_results["distances"][0],
        filtered_results["metadatas"][0]
    ),
    start=1
):
    print(f"Rank: {rank} | Distance: {dist:.4f}")
    print(f"Subject: {meta['subject']}")
    print(doc)
    print()

print("Notice: Only ML documents appear, even though ETL and Pandas might be related.")

Filtered Query: 'how do computers learn from examples'
Filter: Only Machine Learning documents
Rank: 1 | Distance: 0.9956
Subject: Machine Learning
Machine learning models learn patterns from training data

Rank: 2 | Distance: 1.1305
Subject: Machine Learning
Neural networks are inspired by how the human brain works

Notice: Only ML documents appear, even though ETL and Pandas might be related.


In [17]:
print("DISTANCE TO SIMILARITY CONVERSION")
print("=" * 50)
print(f"{'Distance':<15}{'Similarity':<15}{'Interpretation':<20}")
print("=" * 50)

distances = [0.05, 0.20, 0.40, 0.65, 0.90]

interpretations = [
    "Near Identical",
    "Very Similar",
    "Related",
    "Somewhat Related",
    "Different"
]

for dist, interp in zip(distances, interpretations):
    similarity = 1 - dist
    print(f"{dist:<15.2f}{similarity:<15.2f}{interp:<20}")

DISTANCE TO SIMILARITY CONVERSION
Distance       Similarity     Interpretation      
0.05           0.95           Near Identical      
0.20           0.80           Very Similar        
0.40           0.60           Related             
0.65           0.35           Somewhat Related    
0.90           0.10           Different           


In [18]:
import pandas as pd

notes_df = pd.read_csv('college_notes.csv')

print("Dataset loaded!")
print(f"Shape: {notes_df.shape}")
print(f"Columns: {list(notes_df.columns)}")

Dataset loaded!
Shape: (15, 4)
Columns: ['note_id', 'subject', 'topic', 'content']


In [19]:
first_note = notes_df.iloc[0]
print(f"Note ID : {first_note['note_id']}")
print(f"Subject : {first_note['subject']}")
print(f"Topic   : {first_note['topic']}")
print(f"Content : {first_note['content']}")


Note ID : N001
Subject : Data Engineering
Topic   : ETL Pipelines
Content : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.


In [20]:
all_documents = notes_df['content'].tolist()
all_ids = notes_df['note_id'].tolist()

all_metadata = [
    {"subject": row['subject'], "topic": row['topic']}
    for _, row in notes_df.iterrows()
]

print(f"Documents: {len(all_documents)}")
print(f"IDs: {len(all_ids)}")
print(f"Metadata prepared: {len(all_metadata)}")
print()

print("Sample ID:", all_ids[0])
print("Sample Metadata:",all_metadata[0])
print("Sample Document (first 80 chars):", all_documents[0][:80])
print("---")


Documents: 15
IDs: 15
Metadata prepared: 15

Sample ID: N001
Sample Metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
Sample Document (first 80 chars): ETL stands for Extract Transform Load. It is the process of collecting raw data 
---


#  Day 7 - Practice Questions (GenAI + Vector DB Basics)

This README contains answers for Day 7 practice questions covering keyword search, semantic search, embeddings, and ChromaDB usage.

---

##  Q1: Keyword Search vs Semantic Search

**Keyword Search**
- Matches exact words in the query with documents
- Does not understand meaning

**Semantic Search**
- Understands meaning using embeddings
- Finds contextually similar results even if exact words are missing

**Example:**
- Keyword Search: Searching "car" returns documents containing "car"
- Semantic Search: Searching "vehicle for family travel" may return car, SUV, minivan, etc.

---

##  Q2: Output of `model.encode(['hello world'])`

- Converts text into numerical vector (embedding)
- Returns a 2D array or tensor

**Shape:**

Example:
If embedding size = 384 → shape = (1, 384)

---

##  Q3: Model Mismatch Issue

If Model A is used for storing documents and Model B is used for querying:

-  Results will be incorrect or irrelevant
- Reason: Each model creates different vector spaces
- Embeddings are not compatible across models

---

##  Q4: ChromaDB Distance Interpretation

Given distances:

-  Most relevant → 0.12 (smallest distance)
-  Least relevant → 0.87 (largest distance)

---

##  Q5: collection.add() Example

```python
collection.add(
    documents=["Regression notes"],
    metadatas=[{"subject": "ML", "topic": "Regression"}],
    ids=["doc1"]
)
```
## Q6: Filter by Python Programming
```
results = collection.query(
    query_texts=["your search query"],
    n_results=5,
    where={"subject": "Python Programming"}
)
```

# Smart Notes Search Engine - MINIPROJECT 7
Project Overview

Goal: Build a search engine that finds relevant college notes by MEANING, not just keywords

steps:


*   Create new chromaDB collection for college notes
*   Index all 15 notes in the collection
*   Run semantic queries and display top results
*   Filter results by subject
*   Run a comparison: keyword search vs semantic search







In [ ]:

# Step 1: Import Required Libraries
import chromadb
from sentence_transformers import SentenceTransformer


# Step 2: Load Embedding Model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 3: Create ChromaDB Collection
client = chromadb.Client()
collection = client.create_collection(
    name="college_notes"
)



In [22]:
# Step 4: Prepare Notes Dataset
notes = [
    ("Introduction to Python", "Python", "Basics"),
    ("Python loops and conditions", "Python", "Control Flow"),
    ("Functions in Python", "Python", "Functions"),
    ("OOP concepts in C++", "C++", "OOP"),
    ("Constructors and Destructors", "C++", "OOP"),
    ("Pointers in C++", "C++", "Memory"),
    ("DBMS normalization", "DBMS", "Normalization"),
    ("ER diagrams basics", "DBMS", "Design"),
    ("SQL joins explained", "DBMS", "Queries"),
    ("Data structures introduction", "DSA", "Basics"),
    ("Stack and Queue", "DSA", "Linear DS"),
    ("Trees and traversals", "DSA", "Trees"),
    ("OS processes", "OS", "Process Management"),
    ("Deadlock in OS", "OS", "Deadlock"),
    ("Memory management in OS", "OS", "Memory")
]


# Step 5: Extract Documents, Metadata, IDs
documents = [note[0] for note in notes]
metadatas = [
    {
        "subject": note[1],
        "topic": note[2]
    }
    for note in notes
]
ids = [f"note_{i}" for i in range(len(notes))]


# Step 6: Add Notes to ChromaDB Collection
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Successfully indexed {collection.count()} notes.\n")


# Step 7: Semantic Search
print("=" * 60)
print("SEMANTIC SEARCH")
print("=" * 60)
query = "object oriented programming in c++"
results = collection.query(
    query_texts=[query],
    n_results=5
)
for rank, (doc, meta, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\nRank {rank}")
    print(f"Distance : {distance:.4f}")
    print(f"Subject  : {meta['subject']}")
    print(f"Topic    : {meta['topic']}")
    print(f"Document : {doc}")

# Step 8: Metadata Filtering
# Search only inside Python notes
print("\n")
print("=" * 60)
print("FILTERED SEARCH (Python Notes Only)")
print("=" * 60)
filtered_results = collection.query(
    query_texts=["functions and loops"],
    n_results=5,
    where={"subject": "Python"}
)
for rank, doc in enumerate(filtered_results["documents"][0], start=1):
    print(f"{rank}. {doc}")


# Step 9: Traditional Keyword Search
print("\n")
print("=" * 60)
print("KEYWORD SEARCH")
print("=" * 60)
keyword = "python loops"
keyword_results = [
    doc
    for doc in documents
    if "python" in doc.lower()
]
for rank, doc in enumerate(keyword_results, start=1):
    print(f"{rank}. {doc}")


# Step 10: Semantic Search Comparison
print("\n")
print("=" * 60)
print("SEMANTIC SEARCH")
print("=" * 60)
semantic_results = collection.query(
    query_texts=["loops in programming language"],
    n_results=5
)
for rank, doc in enumerate(
    semantic_results["documents"][0],
    start=1
):
    print(f"{rank}. {doc}")

Successfully indexed 15 notes.

SEMANTIC SEARCH

Rank 1
Distance : 0.3833
Subject  : C++
Topic    : OOP
Document : OOP concepts in C++

Rank 2
Distance : 0.9190
Subject  : C++
Topic    : Memory
Document : Pointers in C++

Rank 3
Distance : 1.2468
Subject  : DSA
Topic    : Basics
Document : Data structures introduction

Rank 4
Distance : 1.3388
Subject  : Python
Topic    : Basics
Document : Introduction to Python

Rank 5
Distance : 1.3726
Subject  : OS
Topic    : Memory
Document : Memory management in OS


FILTERED SEARCH (Python Notes Only)
1. Functions in Python
2. Python loops and conditions
3. Introduction to Python


KEYWORD SEARCH
1. Introduction to Python
2. Python loops and conditions
3. Functions in Python


SEMANTIC SEARCH
1. Python loops and conditions
2. OOP concepts in C++
3. Introduction to Python
4. Functions in Python
5. Pointers in C++


--END OF MINIPROJECT DAY-7--